08_pipeline.ipynb
Sprint 2 - PIPE: Integración del pipeline reproducible
En este notebook se integran las transformaciones desarrolladas en las etapas previas del Sprint 2 en un flujo reproducible basado en Pipeline y ColumnTransformer.

Objetivo
Construir y validar un pipeline reutilizable para las siguientes etapas del proyecto, asegurando que las transformaciones puedan aplicarse de forma consistente sobre train, test y datos nuevos.

In [3]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import joblib

# =========================
# 1. Cargar datos
# =========================
X_train = pd.read_csv("../data/processed/X_train_bal.csv")
y_train = pd.read_csv("../data/processed/y_train_bal.csv").values.ravel()

X_test = pd.read_csv("../data/processed/X_test.csv")
y_test = pd.read_csv("../data/processed/y_test.csv").values.ravel()

# =========================
# 2. Separar columnas
# =========================
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns

# =========================
# 3. Pipelines
# =========================
num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# =========================
# 4. ColumnTransformer
# =========================
preprocessor = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols)
])

# =========================
# 5. Pipeline completo
# =========================
model_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", RandomForestClassifier(random_state=42))
])

# =========================
# 6. Entrenamiento
# =========================
model_pipeline.fit(X_train, y_train)

# =========================
# 7. Evaluación
# =========================
y_pred = model_pipeline.predict(X_test)

print(classification_report(y_test, y_pred))

# =========================
# 8. Guardar pipeline
# =========================
joblib.dump(model_pipeline, "pipeline_model.pkl")

print("Pipeline guardado correctamente")

              precision    recall  f1-score   support

           0       0.86      0.87      0.86     12529
           1       0.64      0.62      0.63      4746

    accuracy                           0.80     17275
   macro avg       0.75      0.75      0.75     17275
weighted avg       0.80      0.80      0.80     17275

Pipeline guardado correctamente
